In [ ]:
def vertical_profile_plot(molecule_name, molecule_data, flag_data, date_index = 0):
    """
    Plots a vertical gas profile (VMR against altitude) for inputted molecule ACE-FTS data, quality flags and date index, outlining
    unnatural outliers and instrument errors. Note that unretrieved values and a priori values in the data are removed in the plot.

    Parameters:
    molecule_name (String): Name of the molecule for given data
    molecule_data (Xarray): Xarray for ACE-FTS molecule NetCDF file.
    flag_data (Xarray): Xarray for ACE-FTS molecule quality flags NetCDF file.
    date_index (int, optional)

    Returns:
    matplotlib.figure.Figure: Vertical gas profile plot for the molecule.
    """
    # Setting the 'orbit' and 'sunset_sunrise' parameters from each Xarray as coordinates for the Xarrays
    molecule_data = molecule_data.set_index(index=['orbit', 'sunset_sunrise'])
    flag_data = flag_data.set_index(index=['orbit', 'sunset_sunrise'])

    # Converting the 'hour', 'month', 'day', 'year' parameters of the molecule Xarray into datetime format
    years = molecule_data['year'].values
    months = molecule_data['month'].values
    days = molecule_data['day'].values
    hours = molecule_data['hour'].values
    times = pd.to_datetime({
        'year': years,
        'month': months,
        'day': days,
        'hour': hours
    })

    # Obtaining the date corresponding to the date_time index for the occultation observation
    date = times[date_index].strftime('%Y-%m-%d')

    # Extracting orbit number and sunset_sunrise parameter from the molecule_data Xarray
    orbit_number = molecule_data['orbit'][date_index].values
    ss = molecule_data['sunset_sunrise'][date_index].values

    # Determining the type of occultation (sunset or sunrise) for the observation
    if ss in [0,2]:
        occultation_type = 'Sunset'
    else:
        occultation_type = 'Sunrise'
    
    # Selecting the corresponding quality flags for the occultation observation with orbit number and sunset_sunrise value found above
    flags_profile = flag_data['quality_flag'].sel(orbit = orbit_number, sunset_sunrise = ss).values
    # Extracting the molecule VMR data at all altitudes for the given occultation observation
    molecule_vmr = molecule_data['H2O'][date_index,:].values.flatten()

    # Masking the data for unretrieved values and scaled a priori values
    mask = (flags_profile != 9) & (flags_profile != 8)
    molecule_vmr = molecule_vmr[mask]
    altitude = molecule_data['altitude'].values[mask]
    flags_profile = flags_profile[mask]

    # Creating separate masks for data flagged as unnatural outliers or instrumental errors to be identified on scatter plot
    outlier_mask = (flags_profile == 4) | (flags_profile == 5) | (flags_profile == 6)
    outlier_vmr = molecule_vmr[outlier_mask]
    outlier_altitude = altitude[outlier_mask]
    inst_error_mask = (flags_profile == 7)
    error_vmr = molecule_vmr[inst_error_mask]
    error_altitude = altitude[inst_error_mask]

    # Plotting the data on a scatter plot, circling values which are unnatural outliers or instrumental errors
    plt.figure(figsize = (12,6))
    plt.scatter(altitude, molecule_vmr, label = f'{molecule_name} vertical profile')
    plt.scatter(outlier_altitude, outlier_vmr, facecolors = 'none', edgecolors = 'red', s = 70, linewidths=1.5, label = 'Flagged unnatural outliers')
    plt.scatter(error_altitude, error_vmr, facecolors = 'none', edgecolors='green', s=70, linewidths=1.5, label = 'Flagged instrument errors')
    plt.ylabel(f'{molecule_name} VMR [ppv]')
    plt.xlabel('Altitude [km]')
    plt.title(f'Vertical Profile for {molecule_name} VMR on {date} ({occultation_type})')
    plt.legend()
    plt.show()
    return

In [ ]:
def vertical_profile_plot(
    molecule_name, molecule_data, flag_data, date_index=0, n_surrounding=1
):
    """
    Plots the vertical gas profile (VMR against altitude) for the inputted molecule at the given date_index,
    along with `n_surrounding` profiles before and after it.

    Parameters:
    molecule_name (String): Name of the molecule for the given data.
    molecule_data (Xarray): Xarray for ACE-FTS molecule NetCDF file.
    flag_data (Xarray): Xarray for ACE-FTS molecule quality flags NetCDF file.
    date_index (int): Index of the profile to examine (default is 0).
    n_surrounding (int): Number of profiles before and after the current one to include in the plot.

    Returns:
    None
    """
    # Convert times to datetime for easy comparison
    years = molecule_data['year'].values
    months = molecule_data['month'].values
    days = molecule_data['day'].values
    hours = molecule_data['hour'].values
    times = pd.to_datetime({
        'year': years,
        'month': months,
        'day': days,
        'hour': hours
    })

    # Sort times and indices
    sorted_indices = np.argsort(times.to_numpy())
    sorted_times = times[sorted_indices]

    # Locate the index of the current profile in the sorted array
    current_sorted_index = np.where(sorted_indices == date_index)[0][0]

    # Determine indices for surrounding profiles
    start_index = max(0, current_sorted_index - n_surrounding)
    end_index = min(len(sorted_times), current_sorted_index + n_surrounding + 1)

    surrounding_indices = sorted_indices[start_index:end_index]

    # Helper function to plot a single profile
    def plot_single_profile(index, label_suffix, color, add_legend_labels=False):
        if index is None:
            return  # Skip if no profile exists

        # Extract relevant data
        orbit_number = molecule_data['orbit'][index].values
        ss = molecule_data['sunset_sunrise'][index].values
        if ss in [0,2]:
            occultation_type = 'Sunset'
        else:
            occultation_type = 'Sunrise'
        flags_profile = flag_data['quality_flag'].sel(orbit=orbit_number, sunset_sunrise=ss).values
        molecule_vmr = molecule_data['H2O'][index, :].values.flatten()
        latitude = molecule_data['latitude'][index].values

        # Masking
        mask = (flags_profile != 9) & (flags_profile != 8)
        molecule_vmr = molecule_vmr[mask]
        altitude = molecule_data['altitude'].values[mask]
        flags_profile = flags_profile[mask]

        # Identify flagged values
        outlier_mask = (flags_profile == 4) | (flags_profile == 5) | (flags_profile == 6)
        outlier_vmr = molecule_vmr[outlier_mask]
        outlier_altitude = altitude[outlier_mask]
        inst_error_mask = (flags_profile == 7)
        error_vmr = molecule_vmr[inst_error_mask]
        error_altitude = altitude[inst_error_mask]

        # Plot
        plt.scatter(
            molecule_vmr, altitude,
            label=f"{molecule_name} {label_suffix}, {occultation_type}, latitude = {latitude}",
            color=color, alpha=0.6
        )
        if add_legend_labels:
            plt.scatter(
                outlier_vmr, outlier_altitude,
                marker = '^' ,facecolors="none", edgecolors="red", s=70, linewidths=1.5, label="Flagged unnatural outliers", alpha=0.7
            )
            plt.scatter(
                error_vmr, error_altitude,
                marker = 's',facecolors="none", edgecolors="black", s=70, linewidths=1.5, label="Flagged instrument errors", alpha = 0.7
            )
        else:
            plt.scatter(
                outlier_vmr, outlier_altitude,
                marker = '^',facecolors="none", edgecolors="red", s=70, linewidths=1.5, alpha = 0.7
            )
            plt.scatter(
                error_vmr, error_altitude,
                marker= 's', facecolors="none", edgecolors="black", s=70, linewidths=1.5, alpha = 0.7
            )

    # Create the plot
    plt.figure(figsize=(10, 12))

    # Assign colors to profiles
    colors = plt.cm.viridis(np.linspace(0, 1, len(surrounding_indices)))
    #colors = plt.cm.tab10(np.linspace(0, 1, len(surrounding_indices)))

    # Plot surrounding profiles
    for i, idx in enumerate(surrounding_indices):
        label_suffix = (
            "Current Profile" if idx == date_index else f"Profile {i - n_surrounding}"
        )
        add_legend_labels = (i == len(surrounding_indices)-1)  # Only add legend labels for the last profile
        plot_single_profile(idx, label_suffix, colors[i], add_legend_labels)

    # Add labels, title, and legend
    plt.xlabel(f"{molecule_name} VMR [ppv]")
    plt.ylabel("Altitude [km]")
    plt.xscale("log")
    current_time = times[date_index]
    plt.title(
        f"Vertical Profiles for {molecule_name} on and around {current_time.strftime('%Y-%m-%d %H:%M:%S')}"
    )
    plt.legend()
    plt.show()

# Example usage:
# vertical_profile_plot_with_surrounding("H2O", molecule_data, flag_data, date_index=100, n_surrounding=3)

def preprocess_profile_data(
    molecule_name, molecule_data, flag_data, date_indices, n_surrounding=2, n_altitudes=100
):
    """
    Preprocesses data for training the CNN-LSTM model.

    Parameters:
    - molecule_data: Xarray dataset with molecule data.
    - flag_data: Xarray dataset with quality flags.
    - date_indices: List of indices for profiles to process.
    - n_surrounding: Number of surrounding profiles to include.
    - n_altitudes: Fixed number of altitude levels (for padding/truncation).

    Returns:
    - X: Feature array of shape (n_samples, n_profiles, n_altitudes, n_features).
    - y: Label array of shape (n_samples,).
    """
    features = []
    labels = []

    # Extract time and surrounding profiles
    years = molecule_data['year'].values
    months = molecule_data['month'].values
    days = molecule_data['day'].values
    hours = molecule_data['hour'].values
    times = pd.to_datetime({
        'year': years,
        'month': months,
        'day': days,
        'hour': hours
    })

    # Extract sunset_sunrise values
    sunset_sunrise = molecule_data['sunset_sunrise'].values

    # Define occultation type: 0 or 2 -> Sunset, 1 or 3 -> Sunrise
    occultation_type_value = np.where((sunset_sunrise == 0) | (sunset_sunrise == 2), "Sunset", "Sunrise")

    # Filter date indices to remove profiles with instrumental error flags (7) or missing flag profiles
    filtered_date_indices = filter_date_indices(molecule_data, flag_data, date_indices)


    # Iterate through each profile in the dataset
    for date_index in filtered_date_indices:

        current_occultation = occultation_type_value[date_index]

        # Sort profiles by time
        sorted_indices = np.argsort(times.to_numpy())
        sorted_times = times[sorted_indices]
        sorted_occultation = occultation_type_value[sorted_indices]

        # Locate the index of the current profile in the sorted array
        current_sorted_index = np.where(sorted_indices == date_index)[0][0]

        # Get preceding indices with the same occultation type
        preceding_indices = [
            idx for idx in range(max(0, current_sorted_index - n_surrounding), current_sorted_index + 1)
            if sorted_occultation[idx] == current_occultation
        ]

        surrounding_indices = sorted_indices[preceding_indices]


        # Process profiles
        profile_features = []
        is_bad = False  # Initialize flag for bad profile
        skip_sample = False # Initialize flag for skipping sample if any flag profile is missing
        for idx in surrounding_indices:
            # Extract data for the profile
            orbit_number = molecule_data['orbit'][idx].values
            ss = molecule_data['sunset_sunrise'][idx].values
            try:
                flags_profile = flag_data['quality_flag'].sel(orbit=orbit_number, sunset_sunrise=ss).values
                vmr = molecule_data[molecule_name][idx, :].values.flatten()
                altitude = molecule_data['altitude'].values

                # Mask invalid values
                mask = (flags_profile != 9) & (flags_profile != 8)
                vmr = vmr[mask]
                altitude = altitude[mask]
                flags_profile = flags_profile[mask]

                # Screen for negative VMR values
                positive_mask = vmr >= 0
                vmr = vmr[positive_mask]
                altitude = altitude[positive_mask]
                flags_profile = flags_profile[positive_mask]

                # Normalize VMR, altitude values
                vmr = vmr / np.max(vmr)
                altitude = altitude / np.max(molecule_data['altitude'].values)

                # Identify if the current profile is bad
                if np.any((flags_profile == 4) | (flags_profile == 5) | (flags_profile == 6)):
                    is_bad = True

                # Pad or truncate to fixed altitude levels
                vmr_padded = np.pad(vmr, (0, n_altitudes - len(vmr)), constant_values=-0.1)[:n_altitudes]
                altitude_padded = np.pad(altitude, (0, n_altitudes - len(altitude)), constant_values=-0.1)[:n_altitudes]

                # Encode time features and latitude
                year = molecule_data['year'][idx].values
                month = molecule_data['month'][idx].values
                day = molecule_data['day'][idx].values
                hour = molecule_data['hour'][idx].values
                latitude = molecule_data['latitude'][idx].values/90  # Normalize latitude between -1 and 1
                time_features = encode_time_features(year, month, day, hour)

                # Add indicator for the current profile
                is_current_profile = 1 if idx == date_index else 0
                indicator = np.full(n_altitudes, is_current_profile)

                # Combine features for this profile
                profile_vector = np.column_stack([
                    vmr_padded,             # VMR values
                    altitude_padded,        # Altitude
                    np.full(n_altitudes, latitude),  # Latitude (same for all altitudes)
                    np.tile(time_features, (n_altitudes, 1)),  # Repeat time features for all altitudes
                    indicator  # Indicator for the current profile
                ])
                profile_features.append(profile_vector)
            


            except KeyError:
                print(f"Skipping missing orbit/sunset_sunrise pair: orbit={orbit_number}, sunset_sunrise={ss}")
                skip_sample = True
                continue

        if len(profile_features) < n_surrounding + 1:
            skip_sample = True

        # Skip the sample if any profile is missing
        if skip_sample:
            continue

        # Append features and label for the current profile
        features.append(profile_features)
        labels.append(1 if is_bad else 0)
    

    # Convert to numpy arrays
    X = np.array(features)  # Shape: (n_samples, n_profiles, n_altitudes, n_features)
    y = np.array(labels)    # Shape: (n_samples,)
    return X, y

# Example usage:
# X, y = preprocess_profile_data(h2o_data, h2o_flags, bad_profile_indices, n_surrounding=2, n_altitudes=100)

In [ ]:
features = []
labels = []
max_altitudes = 150

for i in range(flag_data.shape[0]):
    vmr = vmr_data[i,:].values
    altitude = h2o_data['altitude'].values

    orbit_number = h2o_data['orbit'][i].values
    ss = h2o_data['sunset_sunrise'][i].values

    flags = flag_data.sel(orbit = orbit_number, sunset_sunrise = ss).values

    mask = (flags != 8) & (flags != 9)
    vmr = vmr[mask]
    altitude = altitude[mask]
    flags = flags[mask]

    flat_vmr_altitude = np.hstack([np.column_stack((vmr, altitude)).flatten()])
    flat_vmr_altitude_padded = np.pad(flat_vmr_altitude,(0, 2*max_altitudes - len(flat_vmr_altitude)), constant_values=0)
    flags_padded = np.pad(flags, (0, 2*max_altitudes - len(flags)), constant_values=0)
    labels.append(flags_padded)

    year = h2o_data['year'][i].values
    month = h2o_data['month'][i].values
    day = h2o_data['day'][i].values
    hour = h2o_data['hour'][i].values
    time_features = encode_time_features(year,month,day,hour)

    latitude = h2o_data['latitude'][i].values

    feature_vector = np.concatenate([flat_vmr_altitude_padded, [latitude], [ss], time_features])
    features.append(feature_vector)

features_array = np.array(features)
labels_array = np.array(labels)